# Build a Large Language Model from Scratch — Comprehensive Tutorial Notebook - Version 1.0

### Author: Dr. Aliasghar Khavasi (ChatGPT is used in preparation of the notebook)

This notebook is a **learning-first companion tutorial** for understanding and building a GPT-style large language model from scratch in **PyTorch**.

It is inspired by the step-by-step roadmap in **Sebastian Raschka, _Build a Large Language Model (From Scratch)_ (2025)**:
1. understand LLMs at a high level,
2. work with text data,
3. code attention mechanisms,
4. build a GPT model,
5. pretrain it,
6. fine-tune it for classification,
7. fine-tune it to follow instructions,
with optional advanced topics such as **training-loop improvements** and **LoRA**.

## What this notebook tries to do

Instead of only giving you code, this notebook tries to help you answer four questions:

- **What is happening?**  
  We explain each idea in plain language first.

- **Why is it built this way?**  
  We connect code to the design decisions behind LLMs.

- **How do the pieces fit together?**  
  We keep returning to the same pipeline:
  **text -> tokens -> embeddings -> attention -> transformer blocks -> logits -> next-token prediction**.

- **How do I move from toy examples to realistic systems?**  
  We start small and readable, then connect each part to how modern GPT-style models work in practice.

## Learning outcomes

By the end of this notebook, you should be able to:

1. explain what an LLM is and how next-token prediction works;
2. tokenize text, build vocabularies, and create token + positional embeddings;
3. explain self-attention, causal masking, and multi-head attention;
4. build a small GPT-style decoder-only transformer from scratch;
5. pretrain a tiny language model on a toy corpus;
6. generate text with greedy decoding, temperature scaling, and top-k sampling;
7. adapt the pretrained model for **classification**;
8. adapt the pretrained model for **instruction following**;
9. understand why production LLM work often adds features such as warmup schedules and LoRA.

## How to use this notebook

- Read it **top to bottom** the first time.
- Run the code cells as you go.
- Edit the **user input cells** to test your own text, prompts, and instructions.
- Keep in mind that this notebook is designed for **understanding**, not for matching the quality of industrial LLMs.

### Important expectation setting

The model we build here is **tiny** compared with real LLMs.
That is intentional.

A real GPT-like model may use:
- billions of parameters,
- massive tokenized corpora,
- long training runs on many GPUs,
- and carefully tuned training recipes.

This notebook uses the **same core ideas**, but with:
- much smaller models,
- much smaller datasets,
- short CPU-friendly runs,
- and heavily commented code.

## Roadmap of the tutorial

We will follow this progression:

### 1 — What an LLM is
High-level ideas: next-token prediction, pretraining, fine-tuning, transformers, and why decoder-only GPT models work.

### 2 — Working with text data
Tokenization, vocabularies, unknown tokens, sliding-window datasets, embeddings, and positional encodings.

### 3 — Coding attention
Self-attention, queries / keys / values, causal masking, and multi-head attention.

### 4 — Building a GPT model
LayerNorm, feed-forward networks, residual connections, transformer blocks, and the final GPT model.

### 5 — Pretraining
Loss functions, training loops, validation, text generation, checkpointing, and optional library comparisons.

### 6 — Fine-tuning for classification
Turning a pretrained language model into a text classifier.

### 7 — Fine-tuning to follow instructions
Formatting instruction-response pairs and training the model to follow prompts.

### Appendices — Practical upgrades
Learning-rate warmup, cosine decay, gradient clipping, and parameter-efficient fine-tuning with LoRA.

---
# 1 — Understanding Large Language Models

## 1.1 What is an LLM?

A **large language model (LLM)** is a neural network trained to predict the next token in text.

If you give it:

> `The cat sat on the`

the model tries to assign high probability to tokens such as:

- `mat`
- `floor`
- `chair`

The model does this **one step at a time**.

### A useful mental model

Think of an LLM as a very large **autocomplete engine** that has read an enormous amount of text.

But there is an important nuance:

- it does **not** understand in the human sense,
- it does **learn statistical patterns** of language extremely well,
- and those patterns are rich enough to support tasks such as summarization, rewriting, classification, Q&A, and code generation.

### Why this is surprising

The training task sounds simple:

> “Predict the next token.”

Yet this forces the model to learn many things indirectly:

- grammar,
- local word patterns,
- long-range dependencies,
- topic structure,
- style,
- and often a large amount of world knowledge encoded in text.

## 1.2 Generative vs. discriminative thinking

A quick contrast helps:

### Discriminative models
These try to answer questions like:

- Is this email spam?
- Is this review positive or negative?
- Is this image a cat or a dog?

They map **input -> label**.

### Generative models
These try to model how the data itself is structured so they can:

- continue text,
- reconstruct inputs,
- generate answers,
- or produce brand-new samples.

LLMs are **generative text models**.
They learn a probability distribution over token sequences.

## 1.3 The three major stages in practice

A modern GPT-style workflow is usually:

1. **Pretraining**  
   Learn general language patterns from large unlabeled text.

2. **Task adaptation / fine-tuning**  
   Adapt the model to a narrower task:
   - classification,
   - instruction following,
   - domain adaptation,
   - retrieval-aware behavior,
   - or chat alignment.

3. **Inference / deployment**  
   Use the model for text generation or downstream tasks.

### Analogy: education
A helpful analogy is:

- **Pretraining** = general education  
- **Fine-tuning** = specialized professional training  
- **Inference** = doing the actual job

## 1.4 Why transformers replaced older sequence models

Before transformers, many NLP systems relied on **RNNs** and **LSTMs**.

Those models process text step by step and can struggle with very long dependencies.

### Analogy: reading with short-term memory only
Imagine reading a long paragraph but being allowed to remember only a compressed summary of what came before.

That is similar to the bottleneck older sequence models can face.

Transformers improved this by using **attention**:

> when processing a token, the model can directly look back at other tokens that matter.

This is one of the biggest breakthroughs behind modern LLMs.

## 1.5 Decoder-only GPT in one sentence

A GPT-style model is a **decoder-only transformer** trained for **causal next-token prediction**.

That means:

- it reads tokens from left to right,
- it is **not allowed to look into the future**,
- and at each position it predicts what comes next.

This causal masking is why GPT models are so natural for text generation.

## 1.6 A tiny next-token prediction demo

Before we build a neural model, let's build intuition with a tiny **count-based** language model.

This is **not** an LLM.
It is just a small baseline that shows the next-token prediction idea in the simplest possible form.

In [ ]:
from collections import defaultdict, Counter
import random

toy_corpus = (
    "Every effort moves you forward . "
    "Every small step moves you forward . "
    "Practice builds confidence . "
    "Practice builds skill . "
    "Skill grows with practice . "
)

toy_tokens = toy_corpus.split()

bigram_counts = defaultdict(Counter)
for a, b in zip(toy_tokens[:-1], toy_tokens[1:]):
    bigram_counts[a][b] += 1

def sample_next_token(previous_token, temperature=1.0):
    counts = bigram_counts.get(previous_token)
    if not counts:
        return "<unknown>"
    tokens = list(counts.keys())
    freqs = list(counts.values())
    total = sum(freqs)
    probs = [f / total for f in freqs]
    return random.choices(tokens, weights=probs, k=1)[0]

print("Next-token options after 'Practice':", dict(bigram_counts["Practice"]))
print("Next-token options after 'moves':", dict(bigram_counts["moves"]))

start = "Practice"
generated = [start]
for _ in range(6):
    nxt = sample_next_token(generated[-1])
    generated.append(nxt)

print("Toy generated text:", " ".join(generated))

### What this demo teaches

- The core task is already visible: **given previous tokens, guess the next one**.
- But this count-based model is extremely limited:
  - it only remembers immediate neighbors,
  - it has no embeddings,
  - no attention,
  - no abstraction,
  - and no ability to generalize well.

A neural LLM keeps the same high-level objective, but replaces simple counts with:
- embeddings,
- attention,
- deep transformer blocks,
- and gradient-based learning.

---
# Setup and Imports

This notebook uses **PyTorch** for the model implementation.

Optional libraries:
- **tiktoken** for GPT-style byte pair encoding,
- **transformers** for comparing with a library implementation,
- **matplotlib** for visualizations.

If something is missing in your environment, install it in your terminal or notebook environment with commands such as:

```bash
pip install torch matplotlib numpy tiktoken transformers
```

In [ ]:
import math
import re
import time
import copy
import urllib.request
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

---
# 2 — Working with Text Data

## 2.1 Why text has to become numbers

Neural networks do not understand raw words like:

- `"cat"`
- `"transformer"`
- `"attention"`

They operate on numbers.

So we need a pipeline:

1. split text into **tokens**;
2. map tokens to **IDs**;
3. map token IDs to **vectors** (embeddings);
4. add **position information**.

### Analogy: a library catalog
Think of this process like organizing books in a library:

- the **token** is the book title,
- the **token ID** is the catalog number,
- the **embedding** is a dense summary card describing the book's relationships,
- the **position embedding** is the shelf position in a reading sequence.

## 2.2 Loading a small corpus

To keep this notebook runnable on ordinary hardware, we use a tiny text corpus.

If internet access is available, the helper below tries to download **The Verdict** (the public-domain story used in the book examples).
If not, it falls back to a small built-in corpus.

You can also replace this with your own text file later.

In [ ]:
DEFAULT_CORPUS = '''
Every effort moves you forward.
Every small step builds momentum.
Practice improves skill over time.
Attention helps a model decide what matters.
Transformers read text as tokens, not as raw words.
A language model predicts the next token in a sequence.
Pretraining teaches general language patterns.
Fine-tuning adapts the model to a specific task.
Residual connections help deep networks train more easily.
Layer normalization stabilizes activations.
The feed-forward network refines each token independently.
Causal masking prevents the model from peeking at future tokens.
Good evaluation compares training and validation behavior.
A tiny model can still teach the essential ideas behind LLMs.
''' * 6

def load_corpus():
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
    )
    local_path = Path("the-verdict.txt")
    if local_path.exists():
        return local_path.read_text(encoding="utf-8")
    try:
        urllib.request.urlretrieve(url, local_path)
        return local_path.read_text(encoding="utf-8")
    except Exception:
        return DEFAULT_CORPUS

raw_text = load_corpus()
print("Characters in corpus:", len(raw_text))
print(raw_text[:300])

## 2.3 A simple tokenizer from scratch

We begin with a deliberately simple tokenizer to expose the mechanics.

### Analogy: cutting a sentence into Lego pieces
Tokenization is like cutting a sentence into small reusable pieces.

For a simple word-level tokenizer:
- words become pieces,
- punctuation becomes separate pieces,
- unknown words can be mapped to a fallback token like `<|unk|>`.

In [ ]:
class SimpleTokenizer:
    def __init__(self, text, extra_special_tokens=None):
        if extra_special_tokens is None:
            extra_special_tokens = ["<|pad|>", "<|unk|>", "<|bos|>", "<|eos|>"]
        self.special_tokens = extra_special_tokens
        tokens = self._basic_tokenize(text)
        vocab_tokens = sorted(set(tokens))
        self.itos = list(self.special_tokens) + [t for t in vocab_tokens if t not in self.special_tokens]
        self.stoi = {tok: idx for idx, tok in enumerate(self.itos)}
        self.pad_id = self.stoi["<|pad|>"]
        self.unk_id = self.stoi["<|unk|>"]
        self.bos_id = self.stoi["<|bos|>"]
        self.eos_id = self.stoi["<|eos|>"]

    def _basic_tokenize(self, text):
        pieces = re.split(r'([,.;:!?()\"\'\-]|\s+)', text)
        return [p.strip() for p in pieces if p and p.strip()]

    def encode(self, text, add_bos=False, add_eos=False):
        tokens = self._basic_tokenize(text)
        ids = [self.stoi.get(tok, self.unk_id) for tok in tokens]
        if add_bos:
            ids = [self.bos_id] + ids
        if add_eos:
            ids = ids + [self.eos_id]
        return ids

    def decode(self, ids):
        tokens = [self.itos[i] for i in ids]
        text = " ".join(tokens)
        text = re.sub(r'\s+([,.;:!?()\"\'])', r'\1', text)
        text = text.replace(" - ", "-")
        return text

simple_tokenizer = SimpleTokenizer(raw_text)
print("Vocabulary size:", len(simple_tokenizer.itos))

sample_text = "Every effort moves you forward!"
sample_ids = simple_tokenizer.encode(sample_text, add_bos=True, add_eos=True)
print("Sample tokens IDs:", sample_ids)
print("Decoded again:", simple_tokenizer.decode(sample_ids))

## 2.4 Try your own text

Edit the string below and rerun the cell.

The goal is to make the text pipeline feel concrete:
you will see exactly how text is represented before it ever reaches the model.

In [ ]:
YOUR_TEXT = "Transformers made long-range language modeling much easier."

your_ids = simple_tokenizer.encode(YOUR_TEXT, add_bos=True, add_eos=True)
your_tokens = [simple_tokenizer.itos[i] for i in your_ids]

print("Original text:")
print(YOUR_TEXT)
print("\nToken IDs:")
print(your_ids)
print("\nRecovered tokens:")
print(your_tokens)
print("\nDecoded:")
print(simple_tokenizer.decode(your_ids))

## 2.5 Optional: GPT-style byte pair encoding (BPE)

Real GPT models do **not** usually use a simple word-level vocabulary like the one above.

Instead, they use subword tokenization such as **byte pair encoding (BPE)**.

### Why BPE helps
A word-level tokenizer struggles with unknown words.
BPE helps by breaking rare words into smaller pieces that the model has seen before.

### Analogy
If a word tokenizer is like memorizing full Lego structures, BPE is like learning reusable Lego subassemblies.

This is one reason GPT tokenizers are more flexible.

In [ ]:
try:
    import tiktoken
    bpe_tokenizer = tiktoken.get_encoding("gpt2")
    HAVE_TIKTOKEN = True
except Exception as e:
    bpe_tokenizer = None
    HAVE_TIKTOKEN = False
    print("tiktoken not available:", e)

if HAVE_TIKTOKEN:
    bpe_ids = bpe_tokenizer.encode(YOUR_TEXT)
    print("BPE token IDs:", bpe_ids[:30])
    print("Decoded with BPE:", bpe_tokenizer.decode(bpe_ids))
else:
    print("Skipping BPE demo because tiktoken is not installed.")

### Further reading for this section

- Book GitHub repository (code companion)
- OpenAI `tiktoken` repository
- PyTorch tensor and embedding docs

Suggested concepts to review:
- tokenization
- unknown tokens
- subword vocabularies
- embeddings

## 2.6 Sliding-window sampling for next-token prediction

An LLM is trained on many overlapping input-target pairs.

If the token IDs are:

`[10, 11, 12, 13, 14]`

and the context length is 4, we use:

- input: `[10, 11, 12, 13]`
- target: `[11, 12, 13, 14]`

### Analogy: reading through a keyhole
A context window is like looking through a moving keyhole.
You never see the whole book at once.
You only see the current chunk and learn to predict what comes next.

In [ ]:
class LMDataset(Dataset):
    def __init__(self, token_ids, context_length, stride=1):
        self.inputs = []
        self.targets = []
        for i in range(0, len(token_ids) - context_length, stride):
            x = token_ids[i:i + context_length]
            y = token_ids[i + 1:i + context_length + 1]
            self.inputs.append(torch.tensor(x, dtype=torch.long))
            self.targets.append(torch.tensor(y, dtype=torch.long))

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

corpus_ids = simple_tokenizer.encode(raw_text, add_bos=True, add_eos=True)
context_length = 16
dataset = LMDataset(corpus_ids, context_length=context_length, stride=4)

x0, y0 = dataset[0]
print("Input IDs: ", x0.tolist())
print("Target IDs:", y0.tolist())
print("\nInput text:")
print(simple_tokenizer.decode(x0.tolist()))
print("\nTarget text:")
print(simple_tokenizer.decode(y0.tolist()))

## 2.7 Token embeddings and positional embeddings

Now we turn token IDs into vectors.

### Token embeddings
Each token ID gets mapped to a dense vector.
This vector is learned.

### Positional embeddings
Attention alone does not automatically know order.
So we add position information.

### Analogy: names and seat numbers
Imagine a classroom:
- the **token embedding** says **who** a student is,
- the **positional embedding** says **where** they are sitting.

You need both if order matters.

In [ ]:
vocab_size = len(simple_tokenizer.itos)
emb_dim = 32
batch_size_demo = 2

token_embedding = nn.Embedding(vocab_size, emb_dim)
pos_embedding = nn.Embedding(context_length, emb_dim)

demo_loader = DataLoader(dataset, batch_size=batch_size_demo, shuffle=False)
demo_x, demo_y = next(iter(demo_loader))

token_emb = token_embedding(demo_x)
positions = torch.arange(demo_x.shape[1])
pos_emb = pos_embedding(positions)
input_emb = token_emb + pos_emb

print("Batch token IDs shape:", demo_x.shape)
print("Token embedding shape:", token_emb.shape)
print("Positional embedding shape:", pos_emb.shape)
print("Combined input embedding shape:", input_emb.shape)

In [ ]:
plt.figure(figsize=(10, 4))
plt.imshow(input_emb[0].detach().numpy(), aspect="auto")
plt.colorbar()
plt.title("Combined token + positional embeddings for one sequence")
plt.xlabel("Embedding dimension")
plt.ylabel("Token position")
plt.show()

---
# 3 — Coding Attention Mechanisms

## 3.1 Why attention matters

Older sequence models often compressed the past into a single evolving hidden state.

Attention gives the model a better strategy:

> for the current token, decide which earlier tokens matter most.

### Analogy: highlighting while reading
Imagine reading a sentence and using a marker to highlight the few earlier words most relevant to understanding the current word.

That is the basic intuition behind attention.

## 3.2 A toy self-attention example without trainable weights

We start with a tiny hand-worked example.

This is not yet the full GPT mechanism.
It is just a transparent bridge to the real version.

In [ ]:
toy_sentence_tokens = ["Your", "journey", "starts", "with", "one", "step"]
toy_inputs = torch.tensor([
    [0.43, 0.15, 0.89],
    [0.55, 0.87, 0.66],
    [0.57, 0.85, 0.64],
    [0.22, 0.58, 0.33],
    [0.77, 0.25, 0.10],
    [0.05, 0.80, 0.55],
], dtype=torch.float32)

query = toy_inputs[1]  # "journey"
scores = toy_inputs @ query
weights = torch.softmax(scores, dim=0)
context = weights @ toy_inputs

print("Attention scores:", scores)
print("Attention weights:", weights)
print("Context vector for 'journey':", context)

In [ ]:
scores_all = toy_inputs @ toy_inputs.T
weights_all = torch.softmax(scores_all, dim=-1)
contexts_all = weights_all @ toy_inputs

plt.figure(figsize=(6, 5))
plt.imshow(weights_all.detach().numpy(), cmap="viridis")
plt.colorbar()
plt.xticks(range(len(toy_sentence_tokens)), toy_sentence_tokens, rotation=45, ha="right")
plt.yticks(range(len(toy_sentence_tokens)), toy_sentence_tokens)
plt.title("Toy self-attention weights (no trainable projections)")
plt.show()

### What the heatmap means

Each row answers:

> “When building the representation of this token, how much attention is assigned to every token in the sequence?”

A row that puts large weight on earlier words means those earlier words are more influential for the current token representation.

## 3.3 Queries, keys, and values

The trainable version of attention introduces three learned projections:

- **Query**: what the current token is looking for
- **Key**: what each token offers as a match target
- **Value**: the information each token contributes if selected

### Analogy: a research question
Think of attention like using a library:

- the **query** is your research question,
- the **keys** are the catalog entries,
- the **values** are the actual book contents.

A strong query-key match means:
“this source is relevant, so use more of its value.”

In [ ]:
class SingleHeadSelfAttention(nn.Module):
    def __init__(self, d_in, d_out, bias=False):
        super().__init__()
        self.W_q = nn.Linear(d_in, d_out, bias=bias)
        self.W_k = nn.Linear(d_in, d_out, bias=bias)
        self.W_v = nn.Linear(d_in, d_out, bias=bias)

    def forward(self, x, return_weights=False):
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)
        scores = q @ k.transpose(-2, -1)
        weights = torch.softmax(scores / math.sqrt(k.shape[-1]), dim=-1)
        context = weights @ v
        if return_weights:
            return context, weights
        return context

torch.manual_seed(123)
single_attn = SingleHeadSelfAttention(d_in=3, d_out=4)
context_trainable, weights_trainable = single_attn(toy_inputs, return_weights=True)

print("Context shape:", context_trainable.shape)
print("Attention matrix shape:", weights_trainable.shape)

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(weights_trainable.detach().numpy(), cmap="magma")
plt.colorbar()
plt.xticks(range(len(toy_sentence_tokens)), toy_sentence_tokens, rotation=45, ha="right")
plt.yticks(range(len(toy_sentence_tokens)), toy_sentence_tokens)
plt.title("Trainable self-attention weights")
plt.show()

## 3.4 Causal masking

GPT-style models must not look into the future when predicting the next token.

So when the model is at position *t*, it can only attend to positions `<= t`.

### Analogy: writing an exam
Causal masking is like taking an exam where you may use:
- your current question,
- all earlier questions,
- but **not** the answers printed on future pages.

In [ ]:
def causal_mask(size):
    return torch.triu(torch.ones(size, size), diagonal=1).bool()

mask = causal_mask(len(toy_sentence_tokens))
print(mask)

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout=0.0, bias=False):
        super().__init__()
        self.W_q = nn.Linear(d_in, d_out, bias=bias)
        self.W_k = nn.Linear(d_in, d_out, bias=bias)
        self.W_v = nn.Linear(d_in, d_out, bias=bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x, return_weights=False):
        # x shape: [batch, tokens, d_in]
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)

        scores = q @ k.transpose(-2, -1)
        T = x.shape[1]
        scores = scores.masked_fill(self.mask[:T, :T].bool(), float("-inf"))
        weights = torch.softmax(scores / math.sqrt(k.shape[-1]), dim=-1)
        weights = self.dropout(weights)
        context = weights @ v
        if return_weights:
            return context, weights
        return context

toy_batch = toy_inputs.unsqueeze(0)
torch.manual_seed(123)
causal_attn = CausalSelfAttention(d_in=3, d_out=4, context_length=len(toy_sentence_tokens), dropout=0.0)
causal_context, causal_weights = causal_attn(toy_batch, return_weights=True)
print("Causal attention output shape:", causal_context.shape)

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(causal_weights[0].detach().numpy(), cmap="viridis")
plt.colorbar()
plt.xticks(range(len(toy_sentence_tokens)), toy_sentence_tokens, rotation=45, ha="right")
plt.yticks(range(len(toy_sentence_tokens)), toy_sentence_tokens)
plt.title("Causal attention weights (future positions masked)")
plt.show()

## 3.5 Multi-head attention

A single attention head learns one matching pattern.

Multiple heads let the model examine the same sequence in multiple ways at once.

### Analogy: a committee of readers
Imagine several readers analyzing the same paragraph:
- one focuses on grammar,
- one on topic,
- one on long-range references,
- one on style.

Multi-head attention gives the model several such “readers” in parallel.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads, dropout=0.0, bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_q = nn.Linear(d_in, d_out, bias=bias)
        self.W_k = nn.Linear(d_in, d_out, bias=bias)
        self.W_v = nn.Linear(d_in, d_out, bias=bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x, return_weights=False):
        B, T, C = x.shape
        q = self.W_q(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        scores = q @ k.transpose(-2, -1)
        scores = scores.masked_fill(self.mask[:T, :T].bool(), float("-inf"))
        weights = torch.softmax(scores / math.sqrt(self.head_dim), dim=-1)
        weights = self.dropout(weights)

        context = weights @ v
        context = context.transpose(1, 2).contiguous().view(B, T, self.num_heads * self.head_dim)
        output = self.out_proj(context)
        if return_weights:
            return output, weights
        return output

torch.manual_seed(123)
mha = MultiHeadAttention(d_in=3, d_out=8, context_length=len(toy_sentence_tokens), num_heads=2)
mha_out, mha_weights = mha(toy_batch, return_weights=True)
print("Multi-head output shape:", mha_out.shape)
print("Multi-head attention weights shape:", mha_weights.shape)

In [ ]:
fig, axes = plt.subplots(1, mha_weights.shape[1], figsize=(12, 4))
for h in range(mha_weights.shape[1]):
    axes[h].imshow(mha_weights[0, h].detach().numpy(), cmap="plasma")
    axes[h].set_title(f"Head {h}")
    axes[h].set_xticks(range(len(toy_sentence_tokens)))
    axes[h].set_xticklabels(toy_sentence_tokens, rotation=45, ha="right")
    axes[h].set_yticks(range(len(toy_sentence_tokens)))
    axes[h].set_yticklabels(toy_sentence_tokens)
plt.tight_layout()
plt.show()

### Big picture so far

By this point we have assembled the central mechanism of a GPT model:

- token embeddings,
- position information,
- causal self-attention,
- multi-head structure.

Next we will wrap these into full transformer blocks.

---
# 4 — Implementing a GPT Model from Scratch

## 4.1 The remaining building blocks

Attention is only one part of a transformer block.

A typical GPT block also contains:

- **LayerNorm**
- **Feed-forward network (FFN)**
- **Residual / shortcut connections**
- **Dropout**

### Intuition for each part

#### LayerNorm
Keeps activations in a stable range.

**Analogy:** recalibrating the audio levels between songs in a playlist.

#### Feed-forward network
Processes each token independently after attention.

**Analogy:** after the group discussion (attention), each token visits a private expert for extra refinement.

#### Residual connection
Lets information flow through a shortcut path.

**Analogy:** a hallway around a room, so information does not have to pass through every obstacle.

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_hat = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * x_hat + self.shift

class FeedForward(nn.Module):
    def __init__(self, emb_dim, ff_mult=4, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, ff_mult * emb_dim),
            nn.GELU(),
            nn.Linear(ff_mult * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, emb_dim, n_heads, context_length, drop_rate=0.1, qkv_bias=False):
        super().__init__()
        self.norm1 = LayerNorm(emb_dim)
        self.attn = MultiHeadAttention(
            d_in=emb_dim,
            d_out=emb_dim,
            context_length=context_length,
            num_heads=n_heads,
            dropout=drop_rate,
            bias=qkv_bias,
        )
        self.norm2 = LayerNorm(emb_dim)
        self.ff = FeedForward(emb_dim, ff_mult=4, dropout=drop_rate)
        self.drop_resid = nn.Dropout(drop_rate)

    def forward(self, x):
        x = x + self.drop_resid(self.attn(self.norm1(x)))
        x = x + self.drop_resid(self.ff(self.norm2(x)))
        return x

## 4.2 The GPT model

A GPT-style decoder-only model typically does:

1. token embeddings,
2. positional embeddings,
3. a stack of transformer blocks,
4. final layer normalization,
5. linear output head to vocabulary logits.

The output logits tell us how strongly the model prefers each possible next token.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int
    context_length: int
    emb_dim: int = 48
    n_heads: int = 4
    n_layers: int = 1
    drop_rate: float = 0.1
    qkv_bias: bool = False

class GPTModel(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.emb_dim)
        self.pos_emb = nn.Embedding(cfg.context_length, cfg.emb_dim)
        self.drop_emb = nn.Dropout(cfg.drop_rate)

        self.trf_blocks = nn.ModuleList([
            TransformerBlock(
                emb_dim=cfg.emb_dim,
                n_heads=cfg.n_heads,
                context_length=cfg.context_length,
                drop_rate=cfg.drop_rate,
                qkv_bias=cfg.qkv_bias,
            )
            for _ in range(cfg.n_layers)
        ])

        self.final_norm = LayerNorm(cfg.emb_dim)
        self.out_head = nn.Linear(cfg.emb_dim, cfg.vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.shape
        if T > self.cfg.context_length:
            raise ValueError(
            f"Sequence length {T} exceeds model context length {self.cfg.context_length}." 
            "Truncate the input or rebuild the model with a larger context_length. " 
            )
        tok_emb = self.tok_emb(idx)
        pos = torch.arange(T, device=idx.device)
        pos_emb = self.pos_emb(pos)
        x = self.drop_emb(tok_emb + pos_emb)
        for block in self.trf_blocks:
            x = block(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

cfg = GPTConfig(
    vocab_size=vocab_size,
    context_length=context_length,
    emb_dim=48,
    n_heads=4,
    n_layers=1,
    drop_rate=0.1,
    qkv_bias=False,
)

gpt = GPTModel(cfg).to(device)
print(gpt)
print(f"Total parameters: {count_parameters(gpt):,}")

In [ ]:
sample_batch, _ = next(iter(DataLoader(dataset, batch_size=4, shuffle=False)))
sample_batch = sample_batch.to(device)
sample_logits = gpt(sample_batch)

print("Input batch shape:", sample_batch.shape)
print("Logits shape:", sample_logits.shape)
print("Vocabulary size in output layer:", sample_logits.shape[-1])

## 4.3 What the logits mean

For each position in the sequence, the model outputs one vector of length `vocab_size`.

That vector contains **unnormalized scores** for every possible next token.

To turn those scores into probabilities, we apply `softmax`.

The model then chooses or samples the next token from that distribution.

In [ ]:
last_position_logits = sample_logits[0, -1]
probs = torch.softmax(last_position_logits, dim=-1)
topk_probs, topk_ids = torch.topk(probs, k=10)

print("Top-10 next-token guesses:")
for p, i in zip(topk_probs.tolist(), topk_ids.tolist()):
    print(f"{simple_tokenizer.itos[i]:>15s}  prob={p:.4f}")

## 4.4 Generating text from the untrained model

At this point, the model is randomly initialized, so its output should look weak or nonsensical.

That is expected.

This is a useful checkpoint because it shows that the architecture alone is not enough:
the model must **learn** from data.

In [ ]:
def generate_greedy(model, idx, max_new_tokens, context_size):
    model.eval()
    idx = idx.clone()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        next_token_logits = logits[:, -1, :]
        next_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        idx = torch.cat([idx, next_id], dim=1)
    return idx

prompt = "Every effort"
prompt_ids = torch.tensor([simple_tokenizer.encode(prompt, add_bos=True)], dtype=torch.long, device=device)
generated_ids = generate_greedy(gpt, prompt_ids, max_new_tokens=12, context_size=cfg.context_length)
print(simple_tokenizer.decode(generated_ids[0].tolist()))

---
# 5 — Pretraining on Unlabeled Data

## 5.1 The pretraining goal

Pretraining teaches the model to predict the next token over and over again across many contexts.

### Loss function
For next-token prediction, we typically use **cross-entropy loss**.

At each position:
- the model produces logits,
- we compare them with the true next token,
- and we penalize wrong probability distributions.

### Analogy: graded autocomplete
Imagine an autocomplete system being graded after every token:
- if it puts high probability on the true next token, good;
- if it spreads probability badly, it gets penalized.

## 5.2 Preparing train/validation splits

In [ ]:
# Build train/validation split on token IDs
split_idx = int(0.9 * len(corpus_ids))
train_ids = corpus_ids[:split_idx]
val_ids = corpus_ids[split_idx:]

# Large strides keep the notebook fast while still demonstrating the sliding-window idea.
train_dataset = LMDataset(train_ids, context_length=cfg.context_length, stride=8)
val_dataset = LMDataset(val_ids, context_length=cfg.context_length, stride=8)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, drop_last=False)

print("Train sequences:", len(train_dataset))
print("Validation sequences:", len(val_dataset))

## 5.3 Loss and evaluation utilities

We keep these functions simple and explicit so you can see the full logic.

In [ ]:
def calc_batch_loss(model, x, y):
    x = x.to(device)
    y = y.to(device)
    logits = model(x)
    B, T, V = logits.shape
    loss = F.cross_entropy(logits.view(B * T, V), y.view(B * T))
    return loss

@torch.no_grad()
def evaluate_loader(model, loader, max_batches=None):
    model.eval()
    losses = []
    for batch_idx, (x, y) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break
        losses.append(calc_batch_loss(model, x, y).item())
    return float(np.mean(losses)) if losses else float("nan")

## 5.4 Training loop

This is the core pretraining loop.

Key hyperparameters:

- **learning rate**: how large each update step is;
- **batch size**: how many training sequences we process per step;
- **epochs**: how many full passes over the training set;
- **weight decay**: light regularization for the optimizer;
- **dropout**: regularization inside the model.

### Practical interpretation

- learning rate too high -> unstable training
- learning rate too low -> painfully slow progress
- model too small -> underfitting
- model too large for data -> overfitting

In [ ]:
def train_language_model(model, train_loader, val_loader, epochs=2, lr=3e-3, weight_decay=1e-2, eval_every=10):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"step": [], "train_loss": [], "val_loss": []}
    model.train()
    global_step = 0
    start_time = time.time()

    for epoch in range(epochs):
        for x, y in train_loader:
            optimizer.zero_grad()
            loss = calc_batch_loss(model, x, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            if global_step % eval_every == 0:
                train_loss = evaluate_loader(model, train_loader, max_batches=3)
                val_loss = evaluate_loader(model, val_loader, max_batches=3)
                history["step"].append(global_step)
                history["train_loss"].append(train_loss)
                history["val_loss"].append(val_loss)
                print(f"step={global_step:4d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

            global_step += 1

    total_time = time.time() - start_time
    return history, total_time

# Reinitialize a fresh model for training
torch.manual_seed(42)
gpt = GPTModel(cfg).to(device)

history, total_time = train_language_model(
    gpt,
    train_loader,
    val_loader,
    epochs=2,
    lr=3e-3,
    weight_decay=1e-2,
    eval_every=10,
)

print(f"Training time: {total_time:.2f} seconds")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["step"], history["train_loss"], label="Train loss")
plt.plot(history["step"], history["val_loss"], label="Validation loss")
plt.xlabel("Step")
plt.ylabel("Cross-entropy loss")
plt.title("Pretraining loss curves")
plt.legend()
plt.show()

### How to read the loss curves

- If both training and validation loss go down, the model is learning useful structure.
- If training loss drops but validation loss rises, overfitting is starting.
- If both stay high, the model or optimization setup may be too weak.

With tiny datasets, the curves can be noisy.
That is normal.

## 5.5 Better decoding: temperature and top-k

Greedy decoding always picks the highest-probability next token.
That is simple but can become repetitive.

Two standard controls are:

- **temperature**: reshapes confidence
- **top-k**: only sample from the best `k` candidate tokens

### Intuition

- lower temperature -> safer, sharper, more repetitive
- higher temperature -> more diverse, more risky
- smaller top-k -> more conservative
- larger top-k -> more variety

In [ ]:
def sample_next_id(logits, temperature=1.0, top_k=None):
    logits = logits / max(temperature, 1e-6)

    if top_k is not None:
        top_vals, top_idx = torch.topk(logits, top_k)
        masked = torch.full_like(logits, float("-inf"))
        masked.scatter_(dim=-1, index=top_idx, src=top_vals)
        logits = masked

    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)

def generate_text(model, prompt, tokenizer, max_new_tokens=30, temperature=1.0, top_k=None):
    model.eval()
    ids = torch.tensor([tokenizer.encode(prompt, add_bos=True)], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        idx_cond = ids[:, -model.cfg.context_length:]
        with torch.no_grad():
            logits = model(idx_cond)
        next_logits = logits[:, -1, :]
        next_id = sample_next_id(next_logits, temperature=temperature, top_k=top_k)
        ids = torch.cat([ids, next_id], dim=1)
    return tokenizer.decode(ids[0].tolist())

for temp in [0.7, 1.0, 1.3]:
    print("=" * 80)
    print(f"temperature={temp}")
    print(generate_text(gpt, "Every effort", simple_tokenizer, max_new_tokens=15, temperature=temp, top_k=10))

## 5.6 Save and load model weights

Checkpointing is essential in real training workflows.
It lets you:
- resume runs,
- compare checkpoints,
- and separate training from inference.

In [ ]:
checkpoint_path = Path("tiny_gpt_checkpoint.pt")
torch.save(gpt.state_dict(), checkpoint_path)

reloaded_model = GPTModel(cfg).to(device)
reloaded_model.load_state_dict(torch.load(checkpoint_path, map_location=device))

print("Checkpoint saved:", checkpoint_path.exists())
print(generate_text(reloaded_model, "Practice", simple_tokenizer, max_new_tokens=12, temperature=0.8, top_k=10))

## 5.7 Optional: compare with a well-implemented library model

The goal of this notebook is to understand the mechanics from scratch.

But in practice, people often use optimized library implementations.

This optional section shows how a pretrained causal language model can be loaded with **Hugging Face Transformers**.
It is disabled by default because it may require:
- package installation,
- internet access,
- and more memory than the tiny scratch model above.

In [ ]:
RUN_PRETRAINED_LIBRARY_DEMO = False

if RUN_PRETRAINED_LIBRARY_DEMO:
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM

        hf_model_name = "gpt2"
        hf_tokenizer = AutoTokenizer.from_pretrained(hf_model_name)
        hf_model = AutoModelForCausalLM.from_pretrained(hf_model_name).to(device)

        hf_prompt = "Every effort moves you"
        hf_inputs = hf_tokenizer(hf_prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            hf_output = hf_model.generate(
                **hf_inputs,
                max_new_tokens=30,
                do_sample=True,
                top_k=50,
                temperature=0.8,
            )

        print(hf_tokenizer.decode(hf_output[0], skip_special_tokens=True))
    except Exception as e:
        print("Pretrained library demo failed:", e)
else:
    print("Set RUN_PRETRAINED_LIBRARY_DEMO = True to try a pretrained library model.")

---
# 6 — Fine-Tuning for Classification

## 6.1 Why adapt a language model for classification?

A pretrained language model already knows a lot about text structure.
We can reuse that knowledge for supervised tasks.

Examples:
- spam detection
- sentiment analysis
- intent classification
- topic labeling

### Strategy
We keep the language model backbone and add a **classification head** on top.

### Analogy
Pretraining makes someone broadly literate.
Classification fine-tuning teaches them a very specific job:
for example, “read a message and decide whether it is spam.”

## 6.2 A tiny toy spam dataset

Real projects should use a much larger labeled dataset.
Here we keep things intentionally small so you can run everything end to end.

In [ ]:
spam_texts = [
    "Congratulations you won a free prize claim now",
    "Urgent cash reward waiting for you click now",
    "You have been selected for a bonus offer",
    "Free entry in a contest call now",
    "Win money fast by replying today",
    "Exclusive reward available claim your voucher",
]

ham_texts = [
    "Can we meet tomorrow after class",
    "Please send me the report by evening",
    "I will call you after dinner",
    "The meeting starts at nine tomorrow",
    "Thanks for your help with the homework",
    "Let us review the presentation today",
]

clf_texts = spam_texts + ham_texts
clf_labels = [1] * len(spam_texts) + [0] * len(ham_texts)

combined = list(zip(clf_texts, clf_labels))
random.Random(42).shuffle(combined)
clf_texts, clf_labels = zip(*combined)
clf_texts, clf_labels = list(clf_texts), list(clf_labels)

In [ ]:
class ClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.examples = []
        for text, label in zip(texts, labels):
            ids = tokenizer.encode(text, add_bos=True, add_eos=True)[:max_length]
            ids = ids + [tokenizer.pad_id] * (max_length - len(ids))
            self.examples.append((torch.tensor(ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

max_clf_len = 16
split = int(0.8 * len(clf_texts))
clf_train = ClassificationDataset(clf_texts[:split], clf_labels[:split], simple_tokenizer, max_clf_len)
clf_val = ClassificationDataset(clf_texts[split:], clf_labels[split:], simple_tokenizer, max_clf_len)

clf_train_loader = DataLoader(clf_train, batch_size=4, shuffle=True)
clf_val_loader = DataLoader(clf_val, batch_size=4, shuffle=False)

print("Train examples:", len(clf_train))
print("Validation examples:", len(clf_val))

## 6.3 Adding a classification head

A simple approach is:
1. run the token sequence through the GPT backbone,
2. take one summary representation,
3. feed it into a classification layer.

Here we use the **last token representation**.
That is common in decoder-only setups because the later positions have seen the earlier context.

In [ ]:
class GPTForClassification(nn.Module):
    def __init__(self, backbone: GPTModel, num_classes=2, train_backbone=False):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(backbone.cfg.emb_dim, num_classes)

        if not train_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, idx):
        B, T = idx.shape
        tok_emb = self.backbone.tok_emb(idx)
        pos = torch.arange(T, device=idx.device)
        x = self.backbone.drop_emb(tok_emb + self.backbone.pos_emb(pos))
        for block in self.backbone.trf_blocks:
            x = block(x)
        x = self.backbone.final_norm(x)
        last_token = x[:, -1, :]
        logits = self.classifier(last_token)
        return logits

clf_model = GPTForClassification(copy.deepcopy(gpt), num_classes=2, train_backbone=False).to(device)

def eval_classifier(model, loader):
    model.eval()
    losses, correct, total = [], 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            losses.append(loss.item())
            preds = logits.argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += y.numel()
    return float(np.mean(losses)), correct / total if total > 0 else 0.0

def train_classifier(model, train_loader, val_loader, epochs=8, lr=1e-2):
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    history = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()

        train_loss, train_acc = eval_classifier(model, train_loader)
        val_loss, val_acc = eval_classifier(model, val_loader)
        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"epoch={epoch:2d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.3f}")
    return history

clf_history = train_classifier(clf_model, clf_train_loader, clf_val_loader, epochs=8, lr=1e-2)

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(clf_history["epoch"], clf_history["train_loss"], label="Train loss")
ax1.plot(clf_history["epoch"], clf_history["val_loss"], label="Val loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(clf_history["epoch"], clf_history["val_acc"], linestyle="--", label="Val accuracy")
ax2.set_ylabel("Accuracy")
ax2.legend(loc="upper right")

plt.title("Classification fine-tuning")
plt.show()

In [ ]:
def classify_text(text, model, tokenizer, max_length=16):
    ids = tokenizer.encode(text, add_bos=True, add_eos=True)[:max_length]
    ids = ids + [tokenizer.pad_id] * (max_length - len(ids))
    x = torch.tensor([ids], dtype=torch.long, device=device)
    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=-1)
        pred = logits.argmax(dim=-1).item()
    return pred, probs[0].tolist()

NEW_MESSAGE = "You have won a voucher click to claim it now"
pred, probs = classify_text(NEW_MESSAGE, clf_model, simple_tokenizer)
print("Message:", NEW_MESSAGE)
print("Prediction:", "spam" if pred == 1 else "ham")
print("Probabilities [ham, spam]:", [round(p, 4) for p in probs])

### What this section teaches

Fine-tuning for classification is **not** building a brand-new text model from zero.
It is reusing the pretrained backbone and attaching a task-specific head.

Common choices in practice:
- freeze most of the backbone and train only the head,
- unfreeze the last few layers,
- or fine-tune everything if compute and data allow.

---
# 7 — Fine-Tuning to Follow Instructions

## 7.1 What instruction fine-tuning changes

A pretrained model is usually good at text continuation.

But that does **not** automatically mean it follows instructions reliably.

Instruction fine-tuning teaches the model a pattern like:

- here is an instruction,
- here is the expected response,
- imitate this structure.

### Analogy
Pretraining teaches a student to read and write.
Instruction fine-tuning teaches the student to respond to specific assignment prompts.

## 7.2 A tiny instruction dataset

Real instruction tuning uses many thousands to millions of examples.
Here we use a tiny hand-made dataset just to make the mechanics visible.

In [ ]:
instruction_pairs = [
    ("Convert 45 kilometers to meters.", "45 kilometers is 45000 meters."),
    ("Rewrite in passive voice: The artist composed the song.", "The song was composed by the artist."),
    ("Give a synonym for bright.", "A synonym for bright is radiant."),
    ("Correct the grammar: She go to school every day.", "She goes to school every day."),
    ("Summarize: Attention lets a model focus on useful tokens.", "Attention helps a model focus on relevant tokens."),
    ("Answer yes or no: Is water wet?", "Yes."),
    ("Translate to French: hello", "bonjour"),
    ("List two colors.", "Two colors are blue and green."),
]

def format_instruction_example(instruction, response):
    return (
        "<|bos|> Instruction: " + instruction +
        " Response: " + response +
        " <|eos|>"
    )

formatted_instruction_texts = [format_instruction_example(i, r) for i, r in instruction_pairs]
formatted_instruction_texts[:2]

## 7.3 Building labels for supervised instruction fine-tuning

A standard supervised fine-tuning (SFT) trick is:

- feed the whole prompt + response into the model,
- but compute loss mainly on the **response tokens**.

That way, the model learns to produce the answer rather than merely memorize the prompt formatting.

For simplicity, the dataset below masks prompt tokens with `-100`, which tells PyTorch's cross-entropy loss to ignore them.

In [ ]:
class InstructionDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length):
        self.examples = []
        for instruction, response in pairs:
            prompt = "<|bos|> Instruction: " + instruction + " Response:"
            full = prompt + " " + response + " <|eos|>"

            full_ids = tokenizer.encode(full)[:max_length]
            prompt_ids = tokenizer.encode(prompt)[:max_length]

            full_ids = full_ids + [tokenizer.pad_id] * (max_length - len(full_ids))

            labels = full_ids.copy()
            prompt_len = min(len(prompt_ids), max_length)
            for i in range(prompt_len):
                labels[i] = -100  # ignore prompt tokens in the loss

            self.examples.append((
                torch.tensor(full_ids[:-1], dtype=torch.long),
                torch.tensor(labels[1:], dtype=torch.long),
            ))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

inst_max_length = cfg.context_length + 1
inst_dataset = InstructionDataset(instruction_pairs, simple_tokenizer, max_length=inst_max_length)
inst_loader = DataLoader(inst_dataset, batch_size=4, shuffle=True)

x_inst, y_inst = next(iter(inst_loader))
print("Input shape:", x_inst.shape)
print("Labels shape:", y_inst.shape)
print("Decoded input example:")
print(simple_tokenizer.decode(x_inst[0].tolist()))
print("\nLabel IDs example:")
print(y_inst[0].tolist())


## 7.4 Fine-tuning the tiny model on instructions

We start from the pretrained GPT backbone and continue training it on the formatted instruction-response pairs.

In [ ]:
sft_model = copy.deepcopy(gpt).to(device)

def calc_instruction_loss(model, x, y):
    x = x.to(device)
    y = y.to(device)
    logits = model(x)
    B, T, V = logits.shape
    loss = F.cross_entropy(logits.view(B * T, V), y.view(B * T), ignore_index=-100)
    return loss

def train_instruction_model(model, loader, epochs=10, lr=2e-3):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []
        for x, y in loader:
            optimizer.zero_grad()
            loss = calc_instruction_loss(model, x, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            batch_losses.append(loss.item())
        mean_loss = float(np.mean(batch_losses))
        losses.append(mean_loss)
        if epoch % 2 == 0 or epoch == 1:
            print(f"epoch={epoch:2d} | loss={mean_loss:.4f}")
    return losses

instruction_losses = train_instruction_model(sft_model, inst_loader, epochs=10, lr=2e-3)


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(instruction_losses) + 1), instruction_losses)
plt.xlabel("Epoch")
plt.ylabel("Instruction tuning loss")
plt.title("Instruction fine-tuning")
plt.show()


## 7.5 Generating instruction responses

The helper below formats a user instruction into the same pattern used during fine-tuning.

In [ ]:
def respond_to_instruction(model, instruction, tokenizer, max_new_tokens=30, temperature=0.8, top_k=10):
    prompt = "<|bos|> Instruction: " + instruction + " Response:"
    prompt_ids = tokenizer.encode(prompt)[-model.cfg.context_length:]
    ids = torch.tensor([prompt_ids], dtype=torch.long, device=device)

    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = ids[:, -model.cfg.context_length:]
        with torch.no_grad():
            logits = model(idx_cond)
        next_logits = logits[:, -1, :]
        next_id = sample_next_id(next_logits, temperature=temperature, top_k=top_k)
        ids = torch.cat([ids, next_id], dim=1)
        if next_id.item() == tokenizer.eos_id:
            break

    return tokenizer.decode(ids[0].tolist())

USER_INSTRUCTION = "Convert 12 kilometers to meters."
print(respond_to_instruction(sft_model, USER_INSTRUCTION, simple_tokenizer))


### Important realism note

This tiny model and tiny dataset are only for education.
A real instruction-tuned assistant requires:
- far larger datasets,
- better tokenization,
- a stronger pretrained backbone,
- careful prompting templates,
- and much more compute.

But the **core training logic** is already visible here.

---
# A D — Useful Training-Loop Improvements

## Why the basic training loop is usually not enough

In serious training runs, people often add:

- learning-rate warmup,
- cosine decay,
- gradient clipping,
- better checkpointing,
- mixed precision,
- and sometimes gradient accumulation.

These additions do not change the transformer idea itself.
They improve training stability and efficiency.

In [ ]:
def warmup_cosine_schedule(total_steps, warmup_steps, initial_lr, peak_lr, min_lr):
    lrs = []
    for step in range(total_steps):
        if step < warmup_steps:
            lr = initial_lr + (peak_lr - initial_lr) * (step / max(warmup_steps, 1))
        else:
            progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
            cosine = 0.5 * (1 + math.cos(math.pi * progress))
            lr = min_lr + (peak_lr - min_lr) * cosine
        lrs.append(lr)
    return lrs

schedule = warmup_cosine_schedule(
    total_steps=200,
    warmup_steps=30,
    initial_lr=1e-4,
    peak_lr=3e-3,
    min_lr=1e-5,
)

plt.figure(figsize=(8, 4))
plt.plot(schedule)
plt.xlabel("Training step")
plt.ylabel("Learning rate")
plt.title("Warmup + cosine decay schedule")
plt.show()

### How to interpret this schedule

- The **warmup** phase avoids large unstable updates at the start.
- The **cosine decay** phase gradually lowers the learning rate as training progresses.
- This often improves stability and late-stage convergence.

---
# A E — Parameter-Efficient Fine-Tuning with LoRA

##  E.1 Why LoRA exists

Full fine-tuning updates all or most model weights.
That can be expensive.

**LoRA** (Low-Rank Adaptation) inserts small trainable matrices into linear layers so you can adapt a model by training far fewer parameters.

### Analogy
Imagine changing a large machine not by rebuilding the whole machine, but by attaching a small adjustable module at a few critical points.

That is the intuition behind LoRA.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=4, alpha=1.0, bias=True):
        super().__init__()
        self.base = nn.Linear(in_features, out_features, bias=bias)
        self.rank = rank
        self.alpha = alpha

        # LoRA matrices
        self.A = nn.Parameter(torch.randn(in_features, rank) * 0.01)
        self.B = nn.Parameter(torch.zeros(rank, out_features))

        # Freeze base layer to simulate PEFT
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        base_out = self.base(x)
        lora_out = (x @ self.A @ self.B) * (self.alpha / self.rank)
        return base_out + lora_out

demo_layer = LoRALinear(16, 8, rank=2)
x_demo = torch.randn(4, 16)
y_demo = demo_layer(x_demo)

trainable = sum(p.numel() for p in demo_layer.parameters() if p.requires_grad)
total = sum(p.numel() for p in demo_layer.parameters())
print("Output shape:", y_demo.shape)
print(f"Trainable parameters: {trainable:,} / {total:,}")

### Why LoRA is useful

- fewer trainable parameters,
- lower memory cost,
- faster task adaptation,
- easier storage of multiple task-specific adapters.

In real projects, LoRA is often applied to attention projection layers and other linear layers inside transformer blocks.

---
# Final Summary

## What you built in this notebook

You moved through the full conceptual arc of a GPT-style LLM:

1. **High-level LLM ideas**
   - next-token prediction,
   - pretraining,
   - fine-tuning,
   - decoder-only transformers.

2. **Text pipeline**
   - tokenization,
   - token IDs,
   - embeddings,
   - positional embeddings,
   - sliding-window sampling.

3. **Attention**
   - self-attention,
   - causal masking,
   - multi-head attention.

4. **GPT architecture**
   - LayerNorm,
   - feed-forward layers,
   - residual connections,
   - transformer blocks,
   - final language-model head.

5. **Pretraining**
   - cross-entropy loss,
   - training loop,
   - evaluation,
   - sampling strategies,
   - checkpointing.

6. **Fine-tuning**
   - classification adaptation,
   - instruction tuning,
   - response generation.

7. **Advanced upgrades**
   - warmup,
   - cosine decay,
   - LoRA.

## Suggested exercises

1. Increase `emb_dim`, `n_layers`, or `n_heads`.  
   How do parameter count, speed, and training behavior change?

2. Change the tokenizer.  
   What changes when you move from a word-level tokenizer to BPE?

3. Increase `context_length`.  
   Does generation improve?

4. Train longer on a larger corpus.  
   How do the loss curves and samples change?

5. Unfreeze more backbone layers during classification fine-tuning.  
   Does accuracy improve?

6. Expand the instruction dataset and compare generations before and after tuning.

7. Replace a few linear layers with LoRA-style adapters and compare trainable parameter counts.

## Further study

Suggested next topics after this notebook:

- rotary positional embeddings,
- FlashAttention / scaled-dot-product attention implementations,
- RMSNorm vs LayerNorm,
- larger BPE tokenizers,
- checkpoint sharding,
- distributed training,
- evaluation beyond loss,
- RLHF / DPO / preference optimization,
- retrieval-augmented generation,
- quantization and efficient deployment.

You can also compare your scratch implementation with optimized library implementations once the core logic feels natural.

## Final note

A real production LLM is much larger and much harder to train than the one in this notebook.

But if you understand the notebook above, you understand the **core logic** that modern GPT-style models are built from.

That is the real goal.